# MedLoRA · 实验 B2: IU X-Ray 图文 CPT → SLAKE SFT (Kaggle T4)

右侧需挂载数据集 `raddar/chest-xrays-indiana-university` (kernel-metadata 已声明)。

(1) `data/convert_iu_xray.py` 取正位片 + 报告, 最多 5000 份; (2) `configs/cpt_iu_qlora.yaml` 图文 caption 式 CPT 1 epoch; (3) `configs/sft_after_cpt_iu.yaml` 继续 SLAKE SFT; (4) 三张表评估, 与基线 / A / B1 对比。

In [ ]:
REPO_URL = "https://github.com/AugustLoo/MedLoRA.git"
MODEL = "Qwen/Qwen2.5-VL-3B-Instruct"
CPT = "cpt_iu_qlora_r16"
EXP = "sft_after_cpt_iu_r16"
TAG = "cpt_iu_sft_r16"
IU = "/kaggle/input/chest-xrays-indiana-university"

!git clone -q $REPO_URL /kaggle/working/MedLoRA
%cd /kaggle/working/MedLoRA
!mkdir -p /kaggle/temp/raw && rm -rf data/raw && ln -s /kaggle/temp/raw data/raw
!pip install -q -r requirements.txt
!pip install -q "git+https://github.com/hiyouga/LLaMA-Factory.git"
!llamafactory-cli version
!nvidia-smi --query-gpu=name,memory.total --format=csv
!ls $IU && ls $IU/images | head

In [ ]:
!python data/download_slake.py | tail -3
!python data/convert_slake_sharegpt.py
!python data/convert_iu_xray.py --root $IU --max 5000
!python -c "import json; d=json.load(open('data/processed/dataset_info.json')); print(list(d))"

In [ ]:
# 阶段 1: 图文 CPT
!CUDA_VISIBLE_DEVICES=0 llamafactory-cli train configs/cpt_iu_qlora.yaml 2>&1 | grep -v -E "^\s*$|it/s\]|s/it\]" | tail -40
import glob, shutil
for ck in glob.glob(f"outputs/{CPT}/checkpoint-*"): shutil.rmtree(ck)
!ls outputs/$CPT && du -sh outputs/$CPT

In [ ]:
# CPT 效果冒烟: 用 CPT adapter 给 3 张 IU 验证片写报告, 看是否学会了报告语体
import json, sys
sys.path.insert(0, '.')
from medvlm.model import load_model, generate
from PIL import Image
val = json.load(open('data/processed/iu_cpt_val.json'))[:3]
m, p = load_model(MODEL, f'outputs/{CPT}')
for s in val:
    print('GOLD:', s['messages'][1]['content'][:200])
    print('PRED:', generate(m, p, s['messages'][0]['content'].replace('<image>', ''), Image.open(s['images'][0]), 96)[:200])
    print('---')
del m, p
import torch, gc; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# 阶段 2: 在图文 CPT adapter 上继续 SLAKE SFT
!CUDA_VISIBLE_DEVICES=0 llamafactory-cli train configs/sft_after_cpt_iu.yaml 2>&1 | grep -v -E "^\s*$|it/s\]|s/it\]" | tail -60
import json, glob, os, shutil
for name in [CPT, EXP]:
    log = f"outputs/{name}/trainer_log.jsonl"
    if os.path.exists(log):
        rows = [json.loads(l) for l in open(log)]
        tr = [r for r in rows if "loss" in r]; ev = [r for r in rows if "eval_loss" in r]
        print(name, "train loss first/last:", tr[0]["loss"] if tr else None, tr[-1]["loss"] if tr else None,
              "| eval:", [round(r["eval_loss"], 4) for r in ev], "| NaN:", sum(1 for r in tr if r["loss"] != r["loss"]))
for ck in glob.glob(f"outputs/{EXP}/checkpoint-*"): shutil.rmtree(ck)
!du -sh outputs/$CPT outputs/$EXP

In [ ]:
!CUDA_VISIBLE_DEVICES=0 MODEL=$MODEL bash train/eval_all.sh $TAG outputs/$EXP 2>&1 | grep -v -E "it/s\]|s/it\]"

In [ ]:
import json
base = json.load(open("results/baseline_2026-09-15.json"))
a = json.load(open("results/sft_A_2026-09-15.json"))
b1 = json.load(open("results/cpt_sft_B_2026-09-16.json"))
b2 = {k: json.load(open(f"outputs/eval/{k}_{TAG}.json")) for k in ["slake", "textvqa", "pubmedqa"]}
def row(name, *v): print(f"{name:<22}" + "".join(f"{x:>8}" for x in v) + f"{v[-1]-v[1]:>+8.2f}")
print(f"{'metric':<22}{'base':>8}{'A':>8}{'B1':>8}{'B2':>8}{'B2-A':>8}")
for k in ["closed_acc", "open_em", "open_recall", "open_f1"]:
    row("slake_" + k, *[x["slake"]["metrics"][k] for x in (base, a, b1, b2)])
for m in ["X-Ray", "CT", "MRI"]:
    row(f"slake_closed_{m}", *[x["slake"]["by_modality"][m]["closed_acc"] for x in (base, a, b1, b2)])
row("textvqa_acc", *[x["textvqa"]["textvqa_acc"] for x in (base, a, b1, b2)])
row("pubmedqa_acc", *[x["pubmedqa"]["accuracy"] for x in (base, a, b1, b2)])
row("pubmedqa_macro_f1", *[x["pubmedqa"]["macro_f1"] for x in (base, a, b1, b2)])
print("pubmedqa pred_dist:", b2["pubmedqa"]["pred_dist"])
print("RESULTS_JSON", json.dumps(b2))